# 04 — CIT-EmotionNet (CNN–Transformer Interactive network)

**Family:** CNN–Transformer &nbsp;•&nbsp; **Input:** DE features on a topographic grid

A CNN branch (local spatial–spectral patterns over an 8×9 electrode topomap) and a
Transformer branch (global self-attention over the 14 electrode tokens) run in
parallel, then a bidirectional **cross-attention interaction module** lets each
branch attend to the other's representation before fusion and classification.

## 1. Setup

In [ ]:
import os, sys

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.models import cit_emotionnet

set_seed(42)
DEVICE = get_device()

# ----- experiment configuration -----
MODEL_NAME = "CIT_EmotionNet"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> 3-fold / few-epoch smoke test
                      # False -> the full 23-fold benchmark used for the thesis table
print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| QUICK:", QUICK)
print("recipe:", B.REGISTRY[MODEL_NAME])


## 2. Architecture

`src/models/cit_emotionnet.py`. The DE matrix `[14 x 5]` is scattered onto a
`[5 x 8 x 9]` topographic tensor (`de_to_grid`, with a hand-built EPOC electrode
layout — see the `# ASSUMPTION` note). CNN branch: 3 conv blocks → `H*W` tokens.
Transformer branch: per-electrode token embedding + positional parameter + 2
encoder layers. `InteractionModule`: `CNN→Transformer` and `Transformer→CNN`
multi-head cross-attention with residual LayerNorm. Fusion concatenates the mean
of each token stream.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, 5)}
_m = cit_emotionnet.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## Validation protocol — subject-independent (LOSO)

The benchmark uses a single protocol so every architecture is scored identically:
**23-fold leave-one-subject-out** (`data/splits/loso_splits.json`). Fold *k* trains a
fresh model on 22 subjects and tests on the held-out 23rd; normalisation statistics
are fit on the training subjects only and a 10% slice of them is used for early
stopping. We report the **mean ± std of the headline metrics across the 23 folds**.

Everything runs through `src.benchmark_si.run_one` — the one entry point shared by
every model notebook — which reads this model's recipe from `benchmark_si.REGISTRY`,
trains through `src.evaluation.run_loso`, and writes `results/{MODEL_NAME}_results.json`.


## 3. Run the subject-independent (LOSO) benchmark

In [ ]:
# One shared call: 23-fold LOSO for this model, then writes results/CIT_EmotionNet_results.json
res = B.run_one(MODEL_NAME, label=LABEL, quick=QUICK, device=DEVICE)
res

## Notes for the methodology chapter

* Every metric comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* The per-model LOSO recipe (feature view, batch size, epochs, lr) lives in
  `src/benchmark_si.py::REGISTRY` — a single source of truth shared by this
  notebook and the `python -m src.benchmark_si` CLI.
* `params_count` and `train_time_sec` are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` to smoke-test the whole flow (3 folds / few epochs); `LABEL`
  switches the target between valence / arousal / dominance with no other change.
